# dlt Workshop Homework — Instrument an Agent with Logfire, Load Traces with dlt

Homework: [`cohorts/2026/workshops/dlt/homework.md`](../cohorts/2026/workshops/dlt/homework.md)

This rebuilds the Module 1 FAQ agent using Pydantic AI (same code as
[`cohorts/2026/workshops/dlt/homework/agent.py`](../cohorts/2026/workshops/dlt/homework/agent.py) and
[`ingest.py`](../cohorts/2026/workshops/dlt/homework/ingest.py)), instruments it with
[Logfire](https://logfire.pydantic.dev), and answers the three homework questions from real, executed output.

**Q1** and **Q3** are answered directly from spans captured locally via OpenTelemetry — `logfire.instrument_pydantic_ai()`
creates the exact same spans whether or not they're sent to Logfire's cloud, so no account is required for these.
**Q2** additionally requires pulling the traces back out of the real Logfire cloud API via `dlt`, so it needs a
`LOGFIRE_READ_TOKEN` from a real Logfire project.


In [1]:
import os
from dataclasses import dataclass

import requests
from dotenv import load_dotenv

# Root .env holds OPENAI_API_KEY; LOGFIRE_TOKEN / LOGFIRE_READ_TOKEN may live in the hw5 .env instead
load_dotenv()
load_dotenv("llm-zoomcamp-hw5/.env", override=True)

print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("LOGFIRE_TOKEN set:", bool(os.getenv("LOGFIRE_TOKEN")))
print("LOGFIRE_READ_TOKEN set:", bool(os.getenv("LOGFIRE_READ_TOKEN")))


OPENAI_API_KEY set: True
LOGFIRE_TOKEN set: True
LOGFIRE_READ_TOKEN set: False


## Set up the FAQ agent (Module 1, rewritten with Pydantic AI)

Same agent as `cohorts/2026/workshops/dlt/homework/{ingest,agent}.py` — a `minsearch` index over the
course FAQ, wrapped by a Pydantic AI agent with a single `search` tool.


In [2]:
from minsearch import Index


def load_faq_data():
    docs_url = 'https://datatalks.club/faq/json/courses.json'
    response = requests.get(docs_url)
    courses_raw = response.json()

    documents = []
    url_prefix = 'https://datatalks.club/faq'

    for course in courses_raw:
        course_url = f'{url_prefix}{course["path"]}'
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()
        documents.extend(course_data)

    return documents


def build_index(documents):
    index = Index(
        text_fields=['question', 'section', 'answer'],
        keyword_fields=['course']
    )
    index.fit(documents)
    return index


documents = load_faq_data()
index = build_index(documents)
print(f"Loaded {len(documents)} FAQ documents")


Loaded 1380 FAQ documents


In [3]:
from pydantic_ai import Agent, RunContext

INSTRUCTIONS = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


@dataclass
class SearchDeps:
    index: Index


faq_agent = Agent(
    'openai:gpt-5.4-mini',
    deps_type=SearchDeps,
    instructions=INSTRUCTIONS,
)


@faq_agent.tool
def search(ctx: RunContext[SearchDeps], query: str) -> str:
    """Search the FAQ database for entries matching the given query."""
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    results = ctx.deps.index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )
    return results


deps = SearchDeps(index=index)
print("Agent ready")


Agent ready


## Question 1: Instrument the agent with Logfire

> For the query **"How do I run Ollama locally?"**, how many spans does a single agent run produce?
>
> Each span is either the agent run itself, an LLM call, or a tool call. The number can vary between
> runs because the model decides how many times to search.
>
> Options: **1 / 5 / 15 / 30**

We instrument with `logfire.instrument_pydantic_ai()` exactly as the homework specifies, and attach an
`InMemorySpanExporter` so spans can be counted locally and immediately. `send_to_logfire='if-token-present'`
also ships them to the real Logfire project whenever `LOGFIRE_TOKEN` is configured — same code either way.


In [4]:
import logfire
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

span_exporter = InMemorySpanExporter()

logfire.configure(
    send_to_logfire='if-token-present',
    additional_span_processors=[SimpleSpanProcessor(span_exporter)],
    console=False,
)
logfire.instrument_pydantic_ai()

print("Logfire configured. Sending to Logfire cloud:", bool(os.getenv("LOGFIRE_TOKEN")))


Logfire configured. Sending to Logfire cloud: True


In [5]:
QUESTION = "How do I run Ollama locally?"

span_counts = []
last_result = None
for i in range(3):
    span_exporter.clear()
    last_result = await faq_agent.run(QUESTION, deps=deps)
    spans = span_exporter.get_finished_spans()
    span_counts.append(len(spans))
    ops = [s.attributes.get('gen_ai.operation.name') for s in spans]
    tool_calls = ops.count('execute_tool')
    llm_calls = ops.count('chat')
    agent_runs = ops.count('invoke_agent')
    print(f"Run {i + 1}: {len(spans)} spans total  "
          f"(agent run={agent_runs}, llm calls={llm_calls}, tool calls={tool_calls})")

print()
print("Span counts across runs:", span_counts)


Run 1: 4 spans total  (agent run=1, llm calls=2, tool calls=1)


Run 2: 6 spans total  (agent run=1, llm calls=3, tool calls=2)


Run 3: 6 spans total  (agent run=1, llm calls=3, tool calls=2)

Span counts across runs: [4, 6, 6]


**Answer to Q1: 5**

Across 3 real runs of `faq_agent.run("How do I run Ollama locally?")`, span counts were **4, 6, 6**
(1 agent-run span + 2-3 LLM-call spans + 1-2 tool-call spans, depending on how many searches the
model chose to make). These cluster tightly around **5** and are far from the other options (1, 15, 30).

## Question 2: Load traces into DuckDB with dlt

> How many tables did dlt create? Check with:
> ```sql
> SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'agent_traces';
> ```
> Options: **1 / 3 / 24 / 100**

This step pulls the real trace data back out of the Logfire cloud project through its Query API and
loads it into DuckDB with `dlt`, so dlt normalizes the nested span/attribute JSON into tables.


In [6]:
LOGFIRE_READ_TOKEN = os.getenv("LOGFIRE_READ_TOKEN")

if not LOGFIRE_READ_TOKEN:
    print("LOGFIRE_READ_TOKEN not set yet -- skipping the dlt pull for now.")
    print("Once it's available (Logfire project -> Settings -> Read tokens), re-run this cell.")
else:
    print("LOGFIRE_READ_TOKEN found -- ready to build the dlt pipeline.")


LOGFIRE_READ_TOKEN not set yet -- skipping the dlt pull for now.
Once it's available (Logfire project -> Settings -> Read tokens), re-run this cell.


**Answer to Q2:** _(pending — requires a `LOGFIRE_READ_TOKEN` from a real Logfire project)_

## Question 3: Query traces with an agent

> Find the input token usage for the agent run from Q1. Token counts are stored in span attributes as
> `gen_ai.usage.input_tokens`. Sum them across all LLM calls within the trace.
>
> Options: **100-500 / 1500-5000 / 10000-20000 / 50000-100000**

`gen_ai.usage.input_tokens` is set directly on each `chat` span by Pydantic AI's OpenTelemetry
instrumentation, so it can be read straight off the locally captured spans from the last Q1 run —
no Logfire cloud roundtrip needed. We cross-check against Pydantic AI's own `result.usage` for the
same run.


In [7]:
last_run_spans = span_exporter.get_finished_spans()  # spans from the 3rd (last) Q1 run above

llm_call_spans = [s for s in last_run_spans if s.attributes.get('gen_ai.operation.name') == 'chat']
input_tokens_per_call = [s.attributes.get('gen_ai.usage.input_tokens', 0) for s in llm_call_spans]
total_input_tokens = sum(input_tokens_per_call)

print("LLM calls in this trace:", len(llm_call_spans))
print("Input tokens per call:", input_tokens_per_call)
print("Total input tokens (summed across LLM calls):", total_input_tokens)
print()
print("Cross-check via Pydantic AI result.usage:", last_result.usage)


LLM calls in this trace: 3
Input tokens per call: [202, 1281, 2364]
Total input tokens (summed across LLM calls): 3847

Cross-check via Pydantic AI result.usage: RunUsage(input_tokens=3847, cache_read_tokens=1024, output_tokens=276, details={'reasoning_tokens': 0}, requests=3, tool_calls=2)


**Answer to Q3: 1500 - 5000**

For the trace analyzed (Run 3 from Q1, which made 2 searches / 3 LLM calls), input tokens per call
were **[202, 1281, 2364]**, summing to **3847** — matching Pydantic AI's own `result.usage.input_tokens`
exactly (3847). This falls in the **1500-5000** bucket.

## Submission

Submit results here: https://courses.datatalks.club/llm-zoomcamp-2026/homework/dlt
